# Import necessary libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics import roc_curve, auc, classification_report
import time


# ML models

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# DL model

In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

2025-05-02 14:50:29.199394: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746197429.377969      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746197429.431366      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# For visualization

In [4]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Load the dataset

In [5]:
print("Loading dataset...")
df = pd.read_csv('/kaggle/input/malicious-urls-dataset/malicious_phish.csv')
print("Dataset loaded successfully!")

Loading dataset...
Dataset loaded successfully!


# 2. Explore the dataset

In [6]:
print("\n--- Dataset Information ---")
print(f"Shape: {df.shape}")
print("\nFirst 5 rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nClass distribution:")
print(df['type'].value_counts())
print("\nClass distribution percentage:")
print(df['type'].value_counts(normalize=True).mul(100).round(2))
print("\nMissing values:")
print(df.isnull().sum())


--- Dataset Information ---
Shape: (651191, 2)

First 5 rows:
                                                 url        type
0                                   br-icloud.com.br    phishing
1                mp3raid.com/music/krizz_kaliko.html      benign
2                    bopsecrets.org/rexroth/cr/1.htm      benign
3  http://www.garage-pirenne.be/index.php?option=...  defacement
4  http://adventure-nicaragua.net/index.php?optio...  defacement

Data types:
url     object
type    object
dtype: object

Class distribution:
type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64

Class distribution percentage:
type
benign        65.74
defacement    14.81
phishing      14.45
malware        4.99
Name: proportion, dtype: float64

Missing values:
url     0
type    0
dtype: int64


# 3. Data visualization

In [7]:
print("\n--- Creating visualizations ---")

# 3.1 Plot class distribution
plt.figure(figsize=(10, 6))
sns.countplot(x='type', data=df)
plt.title('Distribution of URL Types')
plt.xlabel('URL Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('url_type_distribution.png')
plt.close()

# 3.2 URL length analysis
df['url_length'] = df['url'].apply(len)

plt.figure(figsize=(12, 6))
sns.histplot(data=df, x='url_length', hue='type', kde=True, bins=50)
plt.title('URL Length Distribution by Type')
plt.xlabel('URL Length')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('url_length_distribution.png')
plt.close()

# 3.3 Feature extraction from URLs
def extract_url_features(url):
    features = {}
    features['length'] = len(url)
    features['count_dots'] = url.count('.')
    features['count_hyphens'] = url.count('-')
    features['count_underscores'] = url.count('_')
    features['count_slashes'] = url.count('/')
    features['count_questionmarks'] = url.count('?')
    features['count_equals'] = url.count('=')
    features['count_at'] = url.count('@')
    features['count_and'] = url.count('&')
    features['count_exclamation'] = url.count('!')
    features['count_space'] = url.count(' ')
    features['count_www'] = url.count('www')
    features['count_com'] = url.count('com')
    features['count_digits'] = sum(c.isdigit() for c in url)
    features['has_ip_address'] = 1 if has_ip_pattern(url) else 0
    return features


--- Creating visualizations ---


/usr/local/lib/python3.11/dist-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/usr/local/lib/python3.11/dist-packages/seaborn/_oldcore.py:1075: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  data_subset = grouped_data.get_group(pd_key)
/usr/local/lib/python3.11/dist-packages/seaborn/_oldcore.py:1075: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  data_subset = grouped_data.get_group(pd_key)
/usr/local/lib/python3.11/dist-packages/seaborn/_oldcore.py:1075: FutureWarning: When grouping with a length-1 list-l

In [8]:
def has_ip_pattern(url):
    # Simple check for IP pattern
    import re
    pattern = re.compile(r'(([0-9]|[1-9][0-9]|1[0-9]{2}|2[0-4][0-9]|25[0-5])\.){3}([0-9]|[1-9][0-9]|1[0-9]{2}|2[0-4][0-9]|25[0-5])')
    return bool(pattern.search(url))

In [9]:
# Extract features
print("Extracting URL features...")
url_features = df['url'].apply(extract_url_features).apply(pd.Series)
url_features_df = pd.concat([df['url'], df['type'], url_features], axis=1)

Extracting URL features...


In [10]:
# 3.4 Visualize some of the extracted features by URL type
features_to_plot = ['length', 'count_dots', 'count_slashes', 'count_questionmarks', 'count_digits']
fig, axes = plt.subplots(len(features_to_plot), 1, figsize=(12, 20))

for i, feature in enumerate(features_to_plot):
    sns.boxplot(x='type', y=feature, data=url_features_df, ax=axes[i])
    axes[i].set_title(f'{feature} by URL Type')
    axes[i].set_xlabel('URL Type')
    axes[i].set_ylabel(feature)

plt.tight_layout()
plt.savefig('url_features_by_type.png')
plt.close()

# 4. Prepare data for modeling

In [11]:
# 4.1 Define X and y
X = df['url']
y = df['type']

In [12]:
# 4.2 Encode the target variable
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
print("\nEncoded classes:", encoder.classes_)


Encoded classes: ['benign' 'defacement' 'malware' 'phishing']


In [13]:
# 4.3 Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
print(f"\nTrain set: {X_train.shape}")
print(f"Test set: {X_test.shape}")


Train set: (520952,)
Test set: (130239,)


In [14]:
# 5. Feature extraction using TF-IDF
print("\n--- Feature Extraction using TF-IDF ---")
tfidf_vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=10000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)
print(f"TF-IDF features shape: {X_train_tfidf.shape}")


--- Feature Extraction using TF-IDF ---
TF-IDF features shape: (520952, 10000)


# 4. Function to evaluate models

In [15]:
def evaluate_model(y_true, y_pred, model_name):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    
    print(f"\n--- {model_name} Evaluation ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=encoder.classes_, 
                yticklabels=encoder.classes_)
    plt.title(f'Confusion Matrix - {model_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{model_name}.png')
    plt.close()
    
    # Classification Report
    report = classification_report(y_true, y_pred, target_names=encoder.classes_, output_dict=True)
    return {
        'model_name': model_name,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'report': report
    }


# 6. Model Training and Evaluation

In [16]:
print("\n--- Model Training and Evaluation ---")
model_results = []


--- Model Training and Evaluation ---


###  6.1 Logistic Regression


In [17]:
print("\nTraining Logistic Regression...")
start_time = time.time()
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)
lr_pred = lr_model.predict(X_test_tfidf)
lr_results = evaluate_model(y_test, lr_pred, "Logistic Regression")
model_results.append(lr_results)
print(f"Time taken: {time.time() - start_time:.2f} seconds")


Training Logistic Regression...

--- Logistic Regression Evaluation ---
Accuracy: 0.9648
Precision: 0.9643
Recall: 0.9648
F1 Score: 0.9643
Time taken: 184.94 seconds


### 6.2 Random Forest


In [18]:
print("\nTraining Random Forest...")
start_time = time.time()
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_tfidf, y_train)
rf_pred = rf_model.predict(X_test_tfidf)
rf_results = evaluate_model(y_test, rf_pred, "Random Forest")
model_results.append(rf_results)
print(f"Time taken: {time.time() - start_time:.2f} seconds")


Training Random Forest...

--- Random Forest Evaluation ---
Accuracy: 0.9782
Precision: 0.9780
Recall: 0.9782
F1 Score: 0.9780
Time taken: 3056.07 seconds


###  6.3 XGBoost

In [19]:
print("\nTraining XGBoost...")
start_time = time.time()
xgb_model = XGBClassifier(n_estimators=100, random_state=42)
xgb_model.fit(X_train_tfidf, y_train)
xgb_pred = xgb_model.predict(X_test_tfidf)
xgb_results = evaluate_model(y_test, xgb_pred, "XGBoost")
model_results.append(xgb_results)
print(f"Time taken: {time.time() - start_time:.2f} seconds")


Training XGBoost...

--- XGBoost Evaluation ---
Accuracy: 0.9653
Precision: 0.9648
Recall: 0.9653
F1 Score: 0.9646
Time taken: 556.25 seconds


### 7. Deep Learning model with LSTM


In [20]:
print("\n--- Preparing data for Deep Learning ---")
# Tokenize URLs
max_length = 100
tokenizer = Tokenizer(char_level=True)
tokenizer.fit_on_texts(X_train)
X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_test_sequences = tokenizer.texts_to_sequences(X_test)

# Pad sequences
X_train_padded = pad_sequences(X_train_sequences, maxlen=max_length)
X_test_padded = pad_sequences(X_test_sequences, maxlen=max_length)

# Convert to one-hot encoding for multi-class classification
y_train_categorical = tf.keras.utils.to_categorical(y_train)
y_test_categorical = tf.keras.utils.to_categorical(y_test)

print(f"Vocabulary size: {len(tokenizer.word_index) + 1}")
print(f"Sequence shape: {X_train_padded.shape}")
print(f"One-hot encoded target shape: {y_train_categorical.shape}")

# Build LSTM model
print("\nBuilding LSTM model...")
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 64
lstm_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length),
    LSTM(100),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(y_train_categorical.shape[1], activation='softmax')
])

# Compile
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Summary
lstm_model.summary()

# Train
print("\nTraining LSTM model...")
start_time = time.time()
lstm_history = lstm_model.fit(
    X_train_padded, y_train_categorical,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)
print(f"Time taken: {time.time() - start_time:.2f} seconds")

# Evaluate LSTM
lstm_pred_proba = lstm_model.predict(X_test_padded)
lstm_pred = np.argmax(lstm_pred_proba, axis=1)
lstm_results = evaluate_model(y_test, lstm_pred, "LSTM")
model_results.append(lstm_results)

# Plot training history
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(lstm_history.history['accuracy'])
plt.plot(lstm_history.history['val_accuracy'])
plt.title('LSTM Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.subplot(1, 2, 2)
plt.plot(lstm_history.history['loss'])
plt.plot(lstm_history.history['val_loss'])
plt.title('LSTM Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.tight_layout()
plt.savefig('lstm_training_history.png')
plt.close()


--- Preparing data for Deep Learning ---
Vocabulary size: 272
Sequence shape: (520952, 100)
One-hot encoded target shape: (520952, 4)

Building LSTM model...


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1746201459.921361      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1746201459.922066      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Training LSTM model...
Epoch 1/5


I0000 00:00:1746201464.679732      64 cuda_dnn.cc:529] Loaded cuDNN version 90300


6512/6512 ━━━━━━━━━━━━━━━━━━━━ 55s 8ms/step - accuracy: 0.8581 - loss: 0.4061 - val_accuracy: 0.9433 - val_loss: 0.1706
Epoch 2/5
6512/6512 ━━━━━━━━━━━━━━━━━━━━ 50s 8ms/step - accuracy: 0.9440 - loss: 0.1724 - val_accuracy: 0.9453 - val_loss: 0.1623
Epoch 3/5
6512/6512 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - accuracy: 0.9559 - loss: 0.1382 - val_accuracy: 0.9633 - val_loss: 0.1128
Epoch 4/5
6512/6512 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - accuracy: 0.9658 - loss: 0.1106 - val_accuracy: 0.9705 - val_loss: 0.0957
Epoch 5/5
6512/6512 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - accuracy: 0.9704 - loss: 0.0959 - val_accuracy: 0.9720 - val_loss: 0.0878
Time taken: 253.13 seconds
4070/4070 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step

--- LSTM Evaluation ---
Accuracy: 0.9732
Precision: 0.9730
Recall: 0.9732
F1 Score: 0.9729


# 8. Compare models

In [21]:
print("\n--- Model Comparison ---")
comparison_df = pd.DataFrame({
    'Model': [result['model_name'] for result in model_results],
    'Accuracy': [result['accuracy'] for result in model_results],
    'Precision': [result['precision'] for result in model_results],
    'Recall': [result['recall'] for result in model_results],
    'F1 Score': [result['f1'] for result in model_results]
})
print(comparison_df)

# Plot model comparison
plt.figure(figsize=(12, 8))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
colors = ['blue', 'green', 'red', 'purple']

for i, metric in enumerate(metrics):
    plt.subplot(2, 2, i+1)
    bars = plt.bar(comparison_df['Model'], comparison_df[metric], color=colors[i])
    plt.title(f'Model Comparison - {metric}')
    plt.ylim(0.8, 1.0)  # Adjust as needed
    plt.xticks(rotation=45)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                 f'{height:.4f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('model_comparison.png')
plt.close()


--- Model Comparison ---
                 Model  Accuracy  Precision    Recall  F1 Score
0  Logistic Regression  0.964803   0.964286  0.964803  0.964329
1        Random Forest  0.978194   0.978002  0.978194  0.977996
2              XGBoost  0.965348   0.964840  0.965348  0.964629
3                 LSTM  0.973249   0.973010  0.973249  0.972900


# 9. Feature importance (for Random Forest)

In [22]:
print("\n--- Feature Importance Analysis ---")
# Get feature names
feature_names = tfidf_vectorizer.get_feature_names_out()

# Get top 20 important features
if hasattr(rf_model, 'feature_importances_'):
    importances = rf_model.feature_importances_
    indices = np.argsort(importances)[-20:]  # Top 20 features
    
    plt.figure(figsize=(12, 8))
    plt.title('Top 20 Important Features')
    plt.barh(range(20), importances[indices], color='skyblue')
    plt.yticks(range(20), [feature_names[i] for i in indices])
    plt.xlabel('Relative Importance')
    plt.tight_layout()
    plt.savefig('feature_importance.png')
    plt.close()
    
    print("Top 10 important features:")
    for i in indices[-10:]:
        print(f"{feature_names[i]}: {importances[i]:.4f}")


--- Feature Importance Analysis ---
Top 10 important features:
ttp: 0.0152
tp://: 0.0154
tp:: 0.0155
p:/: 0.0167
http:: 0.0170
http: 0.0194
tp:/: 0.0196
www: 0.0201
ww.: 0.0221
www.: 0.0320


# 10. Best model selection

In [23]:

best_model_idx = comparison_df['F1 Score'].idxmax()
best_model_name = comparison_df.loc[best_model_idx, 'Model']
print(f"\nBest model based on F1 Score: {best_model_name}")
print(f"Best model metrics: {comparison_df.loc[best_model_idx]}")

print("\nProject execution completed successfully!")


Best model based on F1 Score: Random Forest
Best model metrics: Model        Random Forest
Accuracy          0.978194
Precision         0.978002
Recall            0.978194
F1 Score          0.977996
Name: 1, dtype: object

Project execution completed successfully!
